In [1]:
!pip install prometheus_client pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.0 MB/s eta 0:00:00


In [5]:
from prometheus_client import CollectorRegistry, Gauge, push_to_gateway

PUSHGATEWAY_URL = "https://treelike-dacia-fibroplastic.ngrok-free.dev"
JOB_NAME = "colab_llm_benchmark"

def push_benchmark_results(results):
    registry = CollectorRegistry()
    metrics = {
        "ttft":             Gauge("llm_ttft_seconds", "Time To First Token", registry=registry),
        "itl":              Gauge("llm_itl_seconds", "Inter-Token Latency", registry=registry),
        "throughput":       Gauge("llm_throughput_tokens_per_sec", "Tokens per second", registry=registry),
        "request_latency":  Gauge("llm_request_latency_seconds", "Request latency", registry=registry),
        "request_rate":     Gauge("llm_request_rate", "Request rate", registry=registry),
    }
    for key, value in results.items():
        if key in metrics:
            metrics[key].set(value)
    push_to_gateway(PUSHGATEWAY_URL, job=JOB_NAME, registry=registry)
    print(f"✅ Benchmark results pushed → {PUSHGATEWAY_URL}")

# Test push
push_benchmark_results({
    "ttft": 0.43,
    "itl": 0.05,
    "throughput": 312.5,
    "request_latency": 2.1,
    "request_rate": 5.0,
})

✅ Benchmark results pushed → https://treelike-dacia-fibroplastic.ngrok-free.dev


In [3]:
"""
colab_push_metrics.py
─────────────────────
Run this in Google Colab during/after your AIPerf benchmark.
It pushes LLM performance + GPU metrics to your local Pushgateway.

Requirements:
    pip install prometheus_client pynvml
"""

import time
import pynvml
from prometheus_client import CollectorRegistry, Gauge, push_to_gateway

# ── CONFIG ────────────────────────────────────────────────────────────────────
PUSHGATEWAY_URL = "https://treelike-dacia-fibroplastic.ngrok-free.dev"
JOB_NAME        = "colab_llm_benchmark"
PUSH_INTERVAL   = 5                       # seconds between GPU metric pushes
# ─────────────────────────────────────────────────────────────────────────────


def make_registry():
    """Create a fresh registry with all metric definitions."""
    registry = CollectorRegistry()

    metrics = {
        "ttft": Gauge(
            "llm_ttft_seconds",
            "Time To First Token in seconds",
            registry=registry
        ),
        "itl": Gauge(
            "llm_itl_seconds",
            "Inter-Token Latency in seconds",
            registry=registry
        ),
        "throughput": Gauge(
            "llm_throughput_tokens_per_sec",
            "Total tokens per second across all requests",
            registry=registry
        ),
        "request_latency": Gauge(
            "llm_request_latency_seconds",
            "End-to-end latency per request",
            registry=registry
        ),
        "request_rate": Gauge(
            "llm_request_rate",
            "Requests per second sent by AIPerf",
            registry=registry
        ),

        # ── GPU Metrics (from pynvml) ──
        "gpu_util": Gauge(
            "gpu_utilization_percent",
            "GPU compute utilization %",
            registry=registry
        ),
        "gpu_mem_used_mb": Gauge(
            "gpu_memory_used_mb",
            "GPU memory used in MB",
            registry=registry
        ),
        "gpu_mem_total_mb": Gauge(
            "gpu_memory_total_mb",
            "GPU total memory in MB",
            registry=registry
        ),
        "gpu_temp": Gauge(
            "gpu_temperature_celsius",
            "GPU temperature in Celsius",
            registry=registry
        ),
        "gpu_power_watts": Gauge(
            "gpu_power_usage_watts",
            "GPU power draw in Watts",
            registry=registry
        ),
    }
    return registry, metrics


def read_gpu_metrics(handle):
    """Read current GPU stats via pynvml."""
    util      = pynvml.nvmlDeviceGetUtilizationRates(handle)
    mem_info  = pynvml.nvmlDeviceGetMemoryInfo(handle)
    temp      = pynvml.nvmlDeviceGetTemperature(handle, pynvml.NVML_TEMPERATURE_GPU)
    power     = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0  # mW → W

    return {
        "gpu_util":        util.gpu,
        "gpu_mem_used_mb": mem_info.used  / (1024 ** 2),
        "gpu_mem_total_mb":mem_info.total / (1024 ** 2),
        "gpu_temp":        temp,
        "gpu_power_watts": power,
    }


def push_benchmark_results(results: dict):
    """
    Call this once after AIPerf finishes.

    Example:
        push_benchmark_results({
            "ttft":            0.43,
            "itl":             0.05,
            "throughput":      312.5,
            "request_latency": 2.1,
            "request_rate":    5.0,
        })
    """
    registry, metrics = make_registry()

    for key, value in results.items():
        if key in metrics:
            metrics[key].set(value)

    push_to_gateway(PUSHGATEWAY_URL, job=JOB_NAME, registry=registry)
    print(f"✅ Benchmark results pushed → {PUSHGATEWAY_URL}")


def stream_gpu_metrics(duration_seconds: int = 60):
    """
    Stream GPU metrics every PUSH_INTERVAL seconds for `duration_seconds`.
    Run this WHILE AIPerf is running to capture GPU behavior under load.
    """
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)  # GPU 0

    print(f"📡 Streaming GPU metrics to {PUSHGATEWAY_URL} for {duration_seconds}s ...")
    end_time = time.time() + duration_seconds

    while time.time() < end_time:
        registry, metrics = make_registry()
        gpu = read_gpu_metrics(handle)

        for key, value in gpu.items():
            metrics[key].set(value)

        push_to_gateway(PUSHGATEWAY_URL, job=JOB_NAME, registry=registry)
        print(f"  GPU: {gpu['gpu_util']}% util | "
              f"{gpu['gpu_mem_used_mb']:.0f}/{gpu['gpu_mem_total_mb']:.0f} MB | "
              f"{gpu['gpu_temp']}°C | {gpu['gpu_power_watts']:.1f}W")

        time.sleep(PUSH_INTERVAL)

    pynvml.nvmlShutdown()
    print("✅ GPU streaming done.")


# ── HOW TO USE IN COLAB ───────────────────────────────────────────────────────
#
# 1. Stream GPU metrics while benchmark runs (in a background thread):
#
#   import threading
#   t = threading.Thread(target=stream_gpu_metrics, args=(120,))
#   t.start()
#
# 2. Run your AIPerf benchmark here...
#
# 3. Push the AIPerf results after it finishes:
#
#   push_benchmark_results({
#       "ttft":            0.43,
#       "itl":             0.05,
#       "throughput":      312.5,
#       "request_latency": 2.1,
#       "request_rate":    5.0,
#   })
#
# 4. t.join()  ← wait for GPU streaming to finish
# ─────────────────────────────────────────────────────────────────────────────